In [9]:
from pathlib import Path
import pandas as pd

In [6]:
from preprocessing.psychometric_tests.preprocess_psychometric_tests import _reaction_time_accuracy

In [2]:
wcst_file = Path("/home/antonia/Dokumente/Arbeit/multipleye-preprocessing/data/MultiplEYE_ZH_CH_Zurich_1_2025/psychometric-tests-sessions/core_data/WCST/004_ZH_CH_1_PT1/ZHCH1_004_PT1_2025-03-12_20-04-15.csv")

In [24]:
wcst = pd.read_csv(wcst_file)

In [25]:
first_results = _reaction_time_accuracy(wcst,
                                        reaction_time_col="rt",
                                        correctness_col="success")

In [15]:
mean_rt, accuracy, num_items = first_results

In [16]:
mean_rt

np.float64(2.241040928121947)

In [26]:
wcst

,rt,card,chosen card,success,matched on categories,active rule,win streak
0,2.712614,"4,star,green","2,star,green",True,"['shape', 'color']",color,1
1,5.332794,"3,triangle,green","1,triangle,red",False,['shape'],color,0
2,11.461326,"4,circle,green","2,star,green",True,['color'],color,1
3,2.476763,"4,square,green","2,star,green",True,['color'],color,2
4,2.215224,"3,star,red","1,triangle,red",True,['color'],color,3
...,...,...,...,...,...,...,...
59,0.861389,"3,triangle,red","1,triangle,red",True,"['shape', 'color']",color,4
60,1.926816,"3,circle,blue","4,circle,blue",True,"['shape', 'color']",color,5
61,1.048674,"1,square,yellow","3,square,yellow",True,"['shape', 'color']",shape,1
62,5.035326,"2,square,blue","3,square,yellow",True,['shape'],shape,2


In [27]:
wcst["category"] = wcst["active rule"].ne(wcst["active rule"].shift()).cumsum()

In [28]:
wcst

,rt,card,chosen card,success,matched on categories,active rule,win streak,category
0,2.712614,"4,star,green","2,star,green",True,"['shape', 'color']",color,1,1
1,5.332794,"3,triangle,green","1,triangle,red",False,['shape'],color,0,1
2,11.461326,"4,circle,green","2,star,green",True,['color'],color,1,1
3,2.476763,"4,square,green","2,star,green",True,['color'],color,2,1
4,2.215224,"3,star,red","1,triangle,red",True,['color'],color,3,1
...,...,...,...,...,...,...,...,...
59,0.861389,"3,triangle,red","1,triangle,red",True,"['shape', 'color']",color,4,9
60,1.926816,"3,circle,blue","4,circle,blue",True,"['shape', 'color']",color,5,9
61,1.048674,"1,square,yellow","3,square,yellow",True,"['shape', 'color']",shape,1,10
62,5.035326,"2,square,blue","3,square,yellow",True,['shape'],shape,2,10


Number of completed Categories

In [31]:
completed_categories = (wcst["win streak"] == 5).sum()

In [32]:
completed_categories

np.int64(9)

In [33]:
error_percent = 1 - accuracy

In [34]:
error_percent

np.float64(0.203125)

Perseveration

In [35]:
wcst

The history saving thread hit an unexpected error (OperationalError('database or disk is full')).History will not be written to the database.


,rt,card,chosen card,success,matched on categories,active rule,win streak,category
0,2.712614,"4,star,green","2,star,green",True,"['shape', 'color']",color,1,1
1,5.332794,"3,triangle,green","1,triangle,red",False,['shape'],color,0,1
2,11.461326,"4,circle,green","2,star,green",True,['color'],color,1,1
3,2.476763,"4,square,green","2,star,green",True,['color'],color,2,1
4,2.215224,"3,star,red","1,triangle,red",True,['color'],color,3,1
...,...,...,...,...,...,...,...,...
59,0.861389,"3,triangle,red","1,triangle,red",True,"['shape', 'color']",color,4,9
60,1.926816,"3,circle,blue","4,circle,blue",True,"['shape', 'color']",color,5,9
61,1.048674,"1,square,yellow","3,square,yellow",True,"['shape', 'color']",shape,1,10
62,5.035326,"2,square,blue","3,square,yellow",True,['shape'],shape,2,10


In [36]:
rule_map = wcst.drop_duplicates("category").set_index("category")["active rule"]

In [37]:
rule_map

category
1      color
2     number
3      shape
4     number
5      color
6     number
7      color
8      shape
9      color
10     shape
Name: active rule, dtype: str

In [40]:
wcst["previous_rule"] = wcst["category"].sub(1).map(rule_map)

In [41]:
wcst

,rt,card,chosen card,success,matched on categories,active rule,win streak,category,previous_rule
0,2.712614,"4,star,green","2,star,green",True,"['shape', 'color']",color,1,1,NaN
1,5.332794,"3,triangle,green","1,triangle,red",False,['shape'],color,0,1,NaN
2,11.461326,"4,circle,green","2,star,green",True,['color'],color,1,1,NaN
3,2.476763,"4,square,green","2,star,green",True,['color'],color,2,1,NaN
4,2.215224,"3,star,red","1,triangle,red",True,['color'],color,3,1,NaN
...,...,...,...,...,...,...,...,...,...
59,0.861389,"3,triangle,red","1,triangle,red",True,"['shape', 'color']",color,4,9,shape
60,1.926816,"3,circle,blue","4,circle,blue",True,"['shape', 'color']",color,5,9,shape
61,1.048674,"1,square,yellow","3,square,yellow",True,"['shape', 'color']",shape,1,10,color
62,5.035326,"2,square,blue","3,square,yellow",True,['shape'],shape,2,10,color


In [44]:
wcst["perseveration"] = [rule in categories if pd.notna(rule) else False
    for rule, categories in zip(wcst["previous_rule"], wcst["matched on categories"])
]

In [46]:
perseveration_percent = wcst["perseveration"].sum()/num_items

In [48]:
num_items

np.int64(64)

In [47]:
perseveration_percent

np.float64(0.40625)

Perseveration Errors

In [55]:
~wcst["success"]

0     False
1      True
2     False
3     False
4     False
      ...  
59    False
60    False
61    False
62    False
63    False
Name: success, Length: 64, dtype: bool

In [58]:
wcst["perseveration_error"] = (wcst["perseveration"] &  ~wcst["success"])

In [62]:
perseveration_error_percentage = wcst["perseveration_error"].sum()/num_items

In [63]:
perseveration_error_percentage

np.float64(0.125)

More efficient

In [64]:
wcst = pd.read_csv(wcst_file)

In [66]:
mean_rt, accuracy, num_items = _reaction_time_accuracy(wcst, reaction_time_col="rt", correctness_col="success")

In [67]:
error_percent = 1-accuracy

In [68]:
complete_categories = (wcst["win streak"] == 5).sum()

In [72]:
wcst["category index"] = wcst["active rule"].ne(wcst["active rule"].shift()).cumsum()

In [74]:
rule_map = wcst.drop_duplicates("category index").set_index("category index")["active rule"]

In [78]:
previous_rule = (wcst["category index"] - 1).map(rule_map)

In [84]:
wcst["perseveration"] = [pd.notna(rule) and rule in matches for rule, matches in zip(previous_rule, wcst["matched on categories"])]

In [ ]:
perseveration_percent = wcst["perseveration"].sum()/num_items

In [ ]:
perseveration_error_percent = 